In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find MedGemma-27b-text-it project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

os.environ["HF_HOME"] = "/orcd/compute/mghassem/001/gobi1/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/orcd/compute/mghassem/001/gobi1/huggingface"

# Full path to the model snapshot
model_path = "/orcd/compute/mghassem/001/gobi1/huggingface/hub/models--google--medgemma-27b-text-it/snapshots/5b667cf2ddcf064085bc90952edb35a0edbfb79c"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    use_fast=True,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    local_files_only=True
)

prompt = "Give me a short introduction to large language model."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
    do_sample=False
)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|█████████████████████████████| 11/11 [00:07<00:00,  1.38it/s]
The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Okay, here's a short introduction to Large Language Models (LLMs):

**Large Language Models (LLMs) are a type of artificial intelligence (AI) designed to understand, generate, and interact with human language.**

Think of them as incredibly sophisticated pattern-matching machines. They are trained on massive amounts of text data (like books, articles, websites) and learn the statistical relationships between words and concepts.

**Key characteristics:**

*   **"Large":** They have billions (or even trillions) of parameters, which are essentially the variables the model adjusts during training to learn patterns.
*   **"Language":** Their primary function is processing and generating human language.
*   **Capabilities:** They can perform tasks like:
    *   Answering questions
    *   Writing essays, code, or creative content
    *   Translating languages
    *   Summarizing text
    *   Holding conversations (like chatbots)

**In essence, LLMs are powerful tools that can mimic human-lik

In [3]:
import pandas as pd 
import re
import torch

# Load data
df = pd.read_csv(paths.DATA / "14B_High_Low_Irr_Data_Source_.csv")
print("Columns in dataset:")
print(df.columns.tolist())

# Function to extract answer letter using multiple patterns
def extract_answer_letter(text):
    if pd.isna(text) or not text:
        return None
    
    # Try different patterns to extract the answer letter
    patterns = [
        r"Answer:\s*([A-J])",             # "Answer: A"
        r"Answer is\s*([A-J])",           # "Answer is A"
        r"answer is\s*([A-J])",           # "answer is A"
        r"The answer is\s*([A-J])",       # "The answer is A"
        r"the answer is\s*([A-J])",       # "the answer is A"
        r"Option\s*([A-J])",              # "Option A"
        r"option\s*([A-J])",              # "option A"
        r"My answer is\s*([A-J])",        # "My answer is A"
        r"(\n|^)([A-J])\.?\s*$",          # "A." or just "A" at end or newline
        r"select option\s*([A-J])",       # "select option A"
        r"I select\s*([A-J])",            # "I select A"
        r"I choose\s*([A-J])",            # "I choose A"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            # Some patterns have the letter in group 1, others in group 2
            return match.group(1) if len(match.groups()) == 1 else match.group(2)
    
    # If no match found, check if there's a single letter at the end
    words = text.strip().split()
    if words and len(words[-1]) == 1 and words[-1].isalpha() and words[-1].upper() in "ABCDEFGHIJ":
        return words[-1].upper()
    
    return None

# Create results dataframe
results = []

# Loop through the dataset
total_rows = len(df)
print(f"Processing {total_rows} rows...")

for idx, row in df.head(total_rows).iterrows():
    print(f"Processing row {idx+1}/{total_rows}...")
    try:
        context_text = row["High"]
        question = row["question_options"]
        
        # Improved prompt with clearer instructions
        query_full = (
            f"{context_text}\n\n"
            f"{question}\n\n"
            "Based on the information provided, select the correct answer choice (A, B, C, D, etc.).\n\n"
            "IMPORTANT: Your response must end with 'Answer: X' where X is the letter of your chosen option.\n"
            "For example: Answer: A"
    )
    
        # Generate prediction using the model
        inputs = tokenizer(query_full, return_tensors="pt").to(model.device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
        )

        # Decode the generated response
        raw_response = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        
        # Extract the answer letter using improved function
        extracted_answer = extract_answer_letter(raw_response)
        
        # If still no answer found, log more details for debugging
        if extracted_answer is None:
            print(f"⚠️ Could not extract answer from response for row {idx+1}:")
            print(f"Response: {raw_response[:100]}...")
        
        # Create result entry
            qa_id = f"Merge Q{idx + 1}"
        result_entry = {
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_df3", ""),
            "Raw_Response": raw_response,
            "Extracted_Answer": extracted_answer
        }
        
        results.append(result_entry)
        print(f"✅ Processed {qa_id}: Answer = {extracted_answer}")
        
        # Save progress every 10 items (increased frequency for safety)
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(paths.PREDICTIONS / "MedGemma27B_predictions_on_14B_progress.csv", index=False)
            print(f"Saved progress to CSV after {idx+1} items")

    except Exception as e:
        print(f"❌ Error on row {idx}: {str(e)}")
        # Still try to save the entry with error info
        qa_id = f"Merge Q{idx + 1}"
        results.append({
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_df3", ""),
            "Raw_Response": f"ERROR: {str(e)}",
            "Extracted_Answer": None
        })

# Save final results
output_df = pd.DataFrame(results)
output_file = paths.PREDICTIONS / "MedGemma27B_predictions_14B.csv"
output_df.to_csv(output_file, index=False)
print(f"Saved all predictions to {output_file}")

Columns in dataset:
['Origin', 'data_source_df3', 'Patient_Profile', 'Low+Irr', 'High', 'question_options', 'answer_corr']
Processing 1300 rows...
Processing row 1/1300...
⚠️ Could not extract answer from response for row 1:
Response: **Reasoning:**

1.  **Analyze the patient's presentation:** The patient is a 60-year-old woman with ...
✅ Processed Merge Q1: Answer = None
Processing row 2/1300...
⚠️ Could not extract answer from response for row 2:
Response: The patient presents with symptoms suggestive of atypical pneumonia, specifically Mycoplasma pneumon...
✅ Processed Merge Q2: Answer = None
Processing row 3/1300...
✅ Processed Merge Q2: Answer = A
Processing row 4/1300...
✅ Processed Merge Q2: Answer = A
Processing row 5/1300...
✅ Processed Merge Q2: Answer = H
Processing row 6/1300...
✅ Processed Merge Q2: Answer = B
Processing row 7/1300...
✅ Processed Merge Q2: Answer = C
Processing row 8/1300...
✅ Processed Merge Q2: Answer = B
Processing row 9/1300...
✅ Processed Merge Q2: An

⚠️ Could not extract answer from response for row 58:
Response: **Rationale:**

The patient is an 18-year-old woman presenting with pharyngitis and tonsillar exudat...
✅ Processed Merge Q58: Answer = None
Processing row 59/1300...
⚠️ Could not extract answer from response for row 59:
Response: The patient presents with altered consciousness, responds to pain with flexion (suggesting a decreas...
✅ Processed Merge Q59: Answer = None
Processing row 60/1300...
⚠️ Could not extract answer from response for row 60:
Response: **Rationale:**

The patient presents with reticulate hyperpigmentation, which is a key feature of se...
✅ Processed Merge Q60: Answer = None
Saved progress to CSV after 60 items
Processing row 61/1300...
✅ Processed Merge Q60: Answer = D
Processing row 62/1300...
⚠️ Could not extract answer from response for row 62:
Response: **Analysis of the patient's presentation:**

*   **Dyspnea:** The primary symptom.
*   **Smoking His...
✅ Processed Merge Q62: Answer = None
Proce

✅ Processed Merge Q109: Answer = D
Saved progress to CSV after 110 items
Processing row 111/1300...
⚠️ Could not extract answer from response for row 111:
Response: ```

<unused94>thought
The user wants me to identify the most appropriate next step in managing a pa...
✅ Processed Merge Q111: Answer = None
Processing row 112/1300...
⚠️ Could not extract answer from response for row 112:
Response: **Analysis of the case:**

*   **Patient:** 59-year-old man.
*   **Presenting Symptoms:** Nausea, di...
✅ Processed Merge Q112: Answer = None
Processing row 113/1300...
✅ Processed Merge Q112: Answer = C
Processing row 114/1300...
⚠️ Could not extract answer from response for row 114:
Response: **Analysis:**

1.  **Patient:** 6-hour-old newborn boy.
2.  **Presentation:** "Lump on his head" (sw...
✅ Processed Merge Q114: Answer = None
Processing row 115/1300...
✅ Processed Merge Q114: Answer = C
Processing row 116/1300...
✅ Processed Merge Q114: Answer = A
Processing row 117/1300...
⚠️ Could not

⚠️ Could not extract answer from response for row 171:
Response: The patient is a 74-year-old woman with a history of hyperlipidemia. She presents with a 6-month his...
✅ Processed Merge Q171: Answer = None
Processing row 172/1300...
✅ Processed Merge Q171: Answer = C
Processing row 173/1300...
⚠️ Could not extract answer from response for row 173:
Response: ```
Rationale Breakdown:

1.  **Initial Presentation:** The patient is a woman in her 50s with wides...
✅ Processed Merge Q173: Answer = None
Processing row 174/1300...
⚠️ Could not extract answer from response for row 174:
Response: **Rationale:**
The patient is a 50-year-old woman with a recent diagnosis of acute myeloid leukemia ...
✅ Processed Merge Q174: Answer = None
Processing row 175/1300...
✅ Processed Merge Q174: Answer = E
Processing row 176/1300...
⚠️ Could not extract answer from response for row 176:
Response: The patient is a 24-year-old primigravida at 31 weeks' gestation presenting with elevated blood pres...
✅ Pro

✅ Processed Merge Q221: Answer = B
Processing row 223/1300...
✅ Processed Merge Q221: Answer = B
Processing row 224/1300...
✅ Processed Merge Q221: Answer = A
Processing row 225/1300...
⚠️ Could not extract answer from response for row 225:
Response: **Analysis:**

1.  **Patient History:** 52-year-old male, smoker, post-right upper pulmonary lobecto...
✅ Processed Merge Q225: Answer = None
Processing row 226/1300...
⚠️ Could not extract answer from response for row 226:
Response: Here's my thinking process:

1.  **Analyze the patient's presentation:** A 12-year-old boy with feve...
✅ Processed Merge Q226: Answer = None
Processing row 227/1300...
⚠️ Could not extract answer from response for row 227:
Response: The patient presents with symptoms and signs highly suggestive of a breast abscess, a localized coll...
✅ Processed Merge Q227: Answer = None
Processing row 228/1300...
✅ Processed Merge Q227: Answer = C
Processing row 229/1300...
⚠️ Could not extract answer from response for row 

...

✅ Processed Merge Q1023: Answer = A
Processing row 1025/1300...
✅ Processed Merge Q1023: Answer = A
Processing row 1026/1300...
✅ Processed Merge Q1023: Answer = C
Processing row 1027/1300...
⚠️ Could not extract answer from response for row 1027:
Response: The patient is a 5-year-old boy with a history of pica (eating dirt) since age 3. He also has symmet...
✅ Processed Merge Q1027: Answer = None
Processing row 1028/1300...
✅ Processed Merge Q1027: Answer = A
Processing row 1029/1300...
⚠️ Could not extract answer from response for row 1029:
Response: **Reasoning:**

1.  **Identify the core problem:** The patient presents with symptoms (blurry vision...
✅ Processed Merge Q1029: Answer = None
Processing row 1030/1300...
✅ Processed Merge Q1029: Answer = D
Saved progress to CSV after 1030 items
Processing row 1031/1300...
⚠️ Could not extract answer from response for row 1031:
Response: The provided text describes a child with features suggestive of Autism Spectrum Disorder (ASD). The .

⚠️ Could not extract answer from response for row 1082:
Response: **Rationale:**
The patient is a 23-year-old MSM presenting with a 1-month history of skin lesions on...
✅ Processed Merge Q1082: Answer = None
Processing row 1083/1300...
⚠️ Could not extract answer from response for row 1083:
Response: The patient is an adult immigrant with no immunization records. She works as a babysitter and has be...
✅ Processed Merge Q1083: Answer = None
Processing row 1084/1300...
✅ Processed Merge Q1083: Answer = B
Processing row 1085/1300...
⚠️ Could not extract answer from response for row 1085:
Response: The study aims to investigate the association between exposure to contaminated water and the risk of...
✅ Processed Merge Q1085: Answer = None
Processing row 1086/1300...
✅ Processed Merge Q1085: Answer = D
Processing row 1087/1300...
✅ Processed Merge Q1085: Answer = B
Processing row 1088/1300...
⚠️ Could not extract answer from response for row 1088:
Response: The patient is a 14-year-old gi

⚠️ Could not extract answer from response for row 1136:
Response: Here's a thinking process to arrive at the correct answer:

1.  **Analyze the patient's presentation...
✅ Processed Merge Q1136: Answer = None
Processing row 1137/1300...
✅ Processed Merge Q1136: Answer = C
Processing row 1138/1300...
⚠️ Could not extract answer from response for row 1138:
Response: The provided text describes two brothers with chronic bilateral eyelid swelling and irritation since...
✅ Processed Merge Q1138: Answer = None
Processing row 1139/1300...
⚠️ Could not extract answer from response for row 1139:
Response: ```

Rationale Breakdown:

1.  **Identify the core issue:** The question asks about a *false-positiv...
✅ Processed Merge Q1139: Answer = None
Processing row 1140/1300...
✅ Processed Merge Q1139: Answer = B
Saved progress to CSV after 1140 items
Processing row 1141/1300...
⚠️ Could not extract answer from response for row 1141:
Response: **Analysis of the clinical scenario:**

*   **Patient:**

✅ Processed Merge Q1192: Answer = C
Processing row 1194/1300...
✅ Processed Merge Q1192: Answer = C
Processing row 1195/1300...
✅ Processed Merge Q1192: Answer = B
Processing row 1196/1300...
✅ Processed Merge Q1192: Answer = B
Processing row 1197/1300...
⚠️ Could not extract answer from response for row 1197:
Response: The sensitivity of a diagnostic test is defined as the proportion of individuals with the disease wh...
✅ Processed Merge Q1197: Answer = None
Processing row 1198/1300...
✅ Processed Merge Q1197: Answer = C
Processing row 1199/1300...
✅ Processed Merge Q1197: Answer = C
Processing row 1200/1300...
✅ Processed Merge Q1197: Answer = C
Saved progress to CSV after 1200 items
Processing row 1201/1300...
⚠️ Could not extract answer from response for row 1201:
Response: The patient has mild hyperkalemia (potassium 5.4 mEq/L) which has improved from 5.7 mEq/L two weeks ...
✅ Processed Merge Q1201: Answer = None
Processing row 1202/1300...
✅ Processed Merge Q1201: Answer = D
Pro

⚠️ Could not extract answer from response for row 1257:
Response: **Rationale:**

The patient presents with a painful, progressive bulge in the left flank, which is m...
✅ Processed Merge Q1257: Answer = None
Processing row 1258/1300...
⚠️ Could not extract answer from response for row 1258:
Response: The patient presents with symptoms suggestive of olecranon bursitis (redness, pain, swelling around ...
✅ Processed Merge Q1258: Answer = None
Processing row 1259/1300...
✅ Processed Merge Q1258: Answer = A
Processing row 1260/1300...
✅ Processed Merge Q1258: Answer = B
Saved progress to CSV after 1260 items
Processing row 1261/1300...
⚠️ Could not extract answer from response for row 1261:
Response: The patient presents with symptoms suggestive of tricyclic antidepressant (TCA) overdose, specifical...
✅ Processed Merge Q1261: Answer = None
Processing row 1262/1300...
⚠️ Could not extract answer from response for row 1262:
Response: The patient is a 6-year-old male presenting with signs o

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

# Load the data
output_df = pd.read_csv(paths.PREDICTIONS / "MedGemma27B_predictions_14B.csv")
df = pd.read_csv(paths.DATA / "14B_High_Low_Irr_Data_Source_.csv")

# Ensure both dataframes have the same length
assert len(output_df) == len(df), "DataFrames have different lengths!"

# Calculate accuracy (assuming both columns contain the same type of answers to compare)
# Method 1: Exact match
output_df['match'] = (output_df['Extracted_Answer'] == df['answer_corr']).astype(int)

# Overall accuracy statistics
accuracy = output_df['match'].mean()
std_dev = output_df['match'].std()
n = len(output_df)
se = std_dev / np.sqrt(n)  # Standard error
ci_95 = stats.t.interval(0.95, n-1, loc=accuracy, scale=se)

print("=" * 60)
print("OVERALL ACCURACY ANALYSIS")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Standard Deviation: {std_dev:.4f}")
print(f"95% Confidence Interval: [{ci_95[0]:.4f}, {ci_95[1]:.4f}]")
print(f"95% CI (percentage): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")
print(f"Sample Size: {n}")
print("=" * 60)
print()

# Analysis by category (assuming data_source_corr is in one of the dataframes)
# Check which dataframe has data_source_corr
if 'data_source_df3' in output_df.columns:
    analysis_df = output_df.copy()
elif 'data_source_df3' in df.columns:
    analysis_df = output_df.copy()
    analysis_df['data_source_df3'] = df['data_source_df3']
else:
    print("Warning: 'data_source_df3' column not found in either dataframe")
    analysis_df = output_df.copy()

# Category-wise analysis
if 'data_source_df3' in analysis_df.columns:
    category_stats = analysis_df.groupby('data_source_df3')['match'].agg([
        ('Count', 'count'),
        ('Mean_Accuracy', 'mean'),
        ('Std_Dev', 'std'),
        ('SE', lambda x: x.std() / np.sqrt(len(x)))
    ]).reset_index()
    
    # Calculate 95% CI for each category
    ci_lower = []
    ci_upper = []
    
    for idx, row in category_stats.iterrows():
        n_cat = row['Count']
        mean_cat = row['Mean_Accuracy']
        se_cat = row['SE']
        
        if n_cat > 1:
            ci = stats.t.interval(0.95, n_cat-1, loc=mean_cat, scale=se_cat)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        else:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
    
    category_stats['CI_95_Lower'] = ci_lower
    category_stats['CI_95_Upper'] = ci_upper
    
    # Format percentages
    category_stats['Mean_Accuracy_%'] = category_stats['Mean_Accuracy'] * 100
    category_stats['Std_Dev_%'] = category_stats['Std_Dev'] * 100
    category_stats['CI_95_Lower_%'] = category_stats['CI_95_Lower'] * 100
    category_stats['CI_95_Upper_%'] = category_stats['CI_95_Upper'] * 100
    
    print("CATEGORY-WISE ACCURACY ANALYSIS")
    print("=" * 60)
    print(category_stats.to_string(index=False))
    print("=" * 60)
    print()
    
# Create summary statistics table
summary_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Standard Deviation', '95% CI Lower', '95% CI Upper', 'Sample Size'],
    'Value': [f"{accuracy:.4f} ({accuracy*100:.2f}%)", 
              f"{std_dev:.4f}", 
              f"{ci_95[0]:.4f} ({ci_95[0]*100:.2f}%)", 
              f"{ci_95[1]:.4f} ({ci_95[1]*100:.2f}%)", 
              n]
})

print("\nSUMMARY TABLE")
print("=" * 60)
print(summary_table.to_string(index=False))
print("=" * 60)

OVERALL ACCURACY ANALYSIS
Accuracy: 0.3892 (38.92%)
Standard Deviation: 0.4878
95% Confidence Interval: [0.3627, 0.4158]
95% CI (percentage): [36.27%, 41.58%]
Sample Size: 1300

CATEGORY-WISE ACCURACY ANALYSIS
data_source_df3  Count  Mean_Accuracy  Std_Dev       SE  CI_95_Lower  CI_95_Upper  Mean_Accuracy_%  Std_Dev_%  CI_95_Lower_%  CI_95_Upper_%
           jama    582       0.439863 0.496797 0.020593     0.399417     0.480308        43.986254  49.679730      39.941693      48.030816
     medbullets    207       0.439614 0.497543 0.034582     0.371434     0.507793        43.961353  49.754333      37.143419      50.779287
       medxpert    318       0.062893 0.243153 0.013635     0.036066     0.089720         6.289308  24.315317       3.606586       8.972031
           mmlu    193       0.720207 0.450065 0.032396     0.656309     0.784106        72.020725  45.006476      65.630877      78.410574


SUMMARY TABLE
            Metric           Value
  Overall Accuracy 0.3892 (38.92%)
Stan

In [2]:
import pandas as pd
import numpy as np
from scipy import stats

# Load the data
output_df = pd.read_csv(paths.PREDICTIONS / "MedGemma27B_predictions_14B.csv")
df = pd.read_csv(paths.DATA / "14B_High_Low_Irr_Data_Source_.csv")

# Ensure both dataframes have the same length
assert len(output_df) == len(df), "DataFrames have different lengths!"

# Calculate accuracy (assuming both columns contain the same type of answers to compare)
# Method 1: Exact match
output_df['match'] = (output_df['Extracted_Answer'] == df['answer_corr']).astype(int)

# Count None/empty cells
def count_none_empty(series):
    """Count None, NaN, and empty string values"""
    return series.isna().sum() + (series == '').sum() + (series == 'None').sum()

# Overall None/empty counts
extracted_answer_none = count_none_empty(output_df['Extracted_Answer'])
answer_corr_none = count_none_empty(df['answer_corr'])

# Overall accuracy statistics
accuracy = output_df['match'].mean()
std_dev = output_df['match'].std()
n = len(output_df)
se = std_dev / np.sqrt(n)  # Standard error
ci_95 = stats.t.interval(0.95, n-1, loc=accuracy, scale=se)

print("=" * 60)
print("OVERALL ACCURACY ANALYSIS")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Standard Deviation: {std_dev:.4f}")
print(f"95% Confidence Interval: [{ci_95[0]:.4f}, {ci_95[1]:.4f}]")
print(f"95% CI (percentage): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")
print(f"Sample Size: {n}")
print()
print(f"None/Empty in Extracted_Answer: {extracted_answer_none} ({extracted_answer_none/n*100:.2f}%)")
print(f"None/Empty in answer_corr: {answer_corr_none} ({answer_corr_none/n*100:.2f}%)")
print("=" * 60)
print()

# Analysis by category
if 'data_source_df3' in output_df.columns:
    analysis_df = output_df.copy()
elif 'data_source_df3' in df.columns:
    analysis_df = output_df.copy()
    analysis_df['data_source_df3'] = df['data_source_df3']
else:
    print("Warning: 'data_source_df3' column not found in either dataframe")
    analysis_df = output_df.copy()

# Category-wise analysis
if 'data_source_df3' in analysis_df.columns:
    # Add answer_corr to analysis_df for counting
    analysis_df['answer_corr'] = df['answer_corr']
    
    category_stats = analysis_df.groupby('data_source_df3').agg({
        'match': ['count', 'mean', 'std', lambda x: x.std() / np.sqrt(len(x))],
        'Extracted_Answer': lambda x: count_none_empty(x),
        'answer_corr': lambda x: count_none_empty(x)
    })
    
    # Flatten column names
    category_stats.columns = ['Count', 'Mean_Accuracy', 'Std_Dev', 'SE', 
                               'None_Empty_Extracted', 'None_Empty_Answer_Corr']
    category_stats = category_stats.reset_index()
    
    # Calculate 95% CI for each category
    ci_lower = []
    ci_upper = []
    
    for idx, row in category_stats.iterrows():
        n_cat = row['Count']
        mean_cat = row['Mean_Accuracy']
        se_cat = row['SE']
        
        if n_cat > 1:
            ci = stats.t.interval(0.95, n_cat-1, loc=mean_cat, scale=se_cat)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        else:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
    
    category_stats['CI_95_Lower'] = ci_lower
    category_stats['CI_95_Upper'] = ci_upper
    
    # Format percentages
    category_stats['Mean_Accuracy_%'] = category_stats['Mean_Accuracy'] * 100
    category_stats['Std_Dev_%'] = category_stats['Std_Dev'] * 100
    category_stats['CI_95_Lower_%'] = category_stats['CI_95_Lower'] * 100
    category_stats['CI_95_Upper_%'] = category_stats['CI_95_Upper'] * 100
    category_stats['None_Empty_Extracted_%'] = (category_stats['None_Empty_Extracted'] / category_stats['Count']) * 100
    category_stats['None_Empty_Answer_Corr_%'] = (category_stats['None_Empty_Answer_Corr'] / category_stats['Count']) * 100
    
    print("CATEGORY-WISE ACCURACY ANALYSIS")
    print("=" * 80)
    
    # Display main statistics
    display_cols = ['data_source_df3', 'Count', 'Mean_Accuracy_%', 'Std_Dev_%', 
                    'CI_95_Lower_%', 'CI_95_Upper_%']
    print(category_stats[display_cols].to_string(index=False))
    print()
    
    # Display None/Empty counts
    print("CATEGORY-WISE NONE/EMPTY COUNTS")
    print("-" * 80)
    none_cols = ['data_source_df3', 'Count', 'None_Empty_Extracted', 'None_Empty_Extracted_%',
                 'None_Empty_Answer_Corr', 'None_Empty_Answer_Corr_%']
    print(category_stats[none_cols].to_string(index=False))
    print("=" * 80)
    print()
    
# Create summary statistics table
summary_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Standard Deviation', '95% CI Lower', '95% CI Upper', 
               'Sample Size', 'None/Empty Extracted_Answer', 'None/Empty answer_corr'],
    'Value': [f"{accuracy:.4f} ({accuracy*100:.2f}%)", 
              f"{std_dev:.4f}", 
              f"{ci_95[0]:.4f} ({ci_95[0]*100:.2f}%)", 
              f"{ci_95[1]:.4f} ({ci_95[1]*100:.2f}%)", 
              n,
              f"{extracted_answer_none} ({extracted_answer_none/n*100:.2f}%)",
              f"{answer_corr_none} ({answer_corr_none/n*100:.2f}%)"]
})

print("\nSUMMARY TABLE")
print("=" * 60)
print(summary_table.to_string(index=False))
print("=" * 60)

OVERALL ACCURACY ANALYSIS
Accuracy: 0.3892 (38.92%)
Standard Deviation: 0.4878
95% Confidence Interval: [0.3627, 0.4158]
95% CI (percentage): [36.27%, 41.58%]
Sample Size: 1300

None/Empty in Extracted_Answer: 597 (45.92%)
None/Empty in answer_corr: 0 (0.00%)

CATEGORY-WISE ACCURACY ANALYSIS
data_source_df3  Count  Mean_Accuracy_%  Std_Dev_%  CI_95_Lower_%  CI_95_Upper_%
           jama    582        43.986254  49.679730      39.941693      48.030816
     medbullets    207        43.961353  49.754333      37.143419      50.779287
       medxpert    318         6.289308  24.315317       3.606586       8.972031
           mmlu    193        72.020725  45.006476      65.630877      78.410574

CATEGORY-WISE NONE/EMPTY COUNTS
--------------------------------------------------------------------------------
data_source_df3  Count  None_Empty_Extracted  None_Empty_Extracted_%  None_Empty_Answer_Corr  None_Empty_Answer_Corr_%
           jama    582                   210               36.082474 